In [99]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailPulse-Kafka-Streaming")
    .getOrCreate()
)

spark

In [100]:
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "07-01-kafka:9092")
    .option("subscribe", "retailpulse-events")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [101]:
from pyspark.sql import functions as F

events_df = kafka_df.select(
    F.col("value").cast("string").alias("raw_json"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("kafka_timestamp")
)
test_query = (
    events_df.writeStream
    .format("memory")
    .queryName("retailpulse_test")
    .outputMode("append")
    .start()
)
spark.sql("""
    SELECT *
    FROM retailpulse_test
    LIMIT 10
""").show(truncate=False)


+--------+-----+---------+------+---------------+
|raw_json|topic|partition|offset|kafka_timestamp|
+--------+-----+---------+------+---------------+
+--------+-----+---------+------+---------------+



In [102]:
spark.streams.active

In [103]:
spark.sql("""
    SELECT *
    FROM retailpulse_test
    LIMIT 10
""").show(truncate=False)

+--------+-----+---------+------+---------------+
|raw_json|topic|partition|offset|kafka_timestamp|
+--------+-----+---------+------+---------------+
+--------+-----+---------+------+---------------+



In [104]:
spark.sql("""
    SELECT COUNT(*)
    FROM retailpulse_test
""").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



In [105]:
test_query.stop()

In [106]:
spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")

'hdfs://namenode:8020'

In [107]:
BRONZE_PATH = "hdfs://namenode:8020/data/retailpulse/bronze/streaming/events"
CHECKPOINT_PATH = "hdfs://namenode:8020/data/retailpulse/checkpoints/streaming/events"

print(BRONZE_PATH)
print(CHECKPOINT_PATH)

hdfs://namenode:8020/data/retailpulse/bronze/streaming/events
hdfs://namenode:8020/data/retailpulse/checkpoints/streaming/events


In [108]:
events_df
bronze_query = (
    events_df.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", BRONZE_PATH)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .start()
)

In [109]:
spark.streams.active

In [110]:
bronze_query.status

{'message': 'Getting offsets from KafkaV2[Subscribe[retailpulse-events]]',
 'isDataAvailable': False,
 'isTriggerActive': True}

In [111]:
bronze_df = spark.read.parquet(BRONZE_PATH)

bronze_df.count()

20000

In [112]:
bronze_query.stop()

In [113]:
spark.streams.active

[]

In [114]:
BRONZE_CSV_PATH = "hdfs://namenode:8020/data/retailpulse/bronze/streaming/events_csv"

CHECKPOINT_CSV_PATH = "hdfs://namenode:8020/data/retailpulse/checkpoints/streaming/events_csv"

In [115]:
bronze_csv_query = (
    events_df.writeStream
    .format("csv")
    .outputMode("append")
    .option("path", BRONZE_CSV_PATH)
    .option("checkpointLocation", CHECKPOINT_CSV_PATH)
    .option("header", "true")
    .start()
)

In [116]:
bronze_query.stop()

In [117]:
spark.streams.active

In [118]:
for query in spark.streams.active:
    query.stop()

In [119]:
spark.streams.active

[]

In [120]:
BRONZE_CSV_PATH = "hdfs://namenode:8020/data/retailpulse/bronze/streaming/events_csv"

CHECKPOINT_CSV_PATH = "hdfs://namenode:8020/data/retailpulse/checkpoints/streaming/events_csv"

In [121]:
bronze_csv_query = (
    events_df.writeStream
    .format("csv")
    .outputMode("append")
    .option("path", BRONZE_CSV_PATH)
    .option("checkpointLocation", CHECKPOINT_CSV_PATH)
    .option("header", "true")
    .start()
)

In [122]:
for query in spark.streams.active:
    query.stop()

spark.streams.active

[]

In [123]:
BRONZE_CSV_PATH = "hdfs://namenode:8020/data/retailpulse/bronze/streaming/events_csv"

CHECKPOINT_CSV_PATH = "hdfs://namenode:8020/data/retailpulse/checkpoints/streaming/events_csv"

In [124]:
bronze_csv_query = (
    events_df.writeStream
    .format("csv")
    .outputMode("append")
    .option("path", BRONZE_CSV_PATH)
    .option("checkpointLocation", CHECKPOINT_CSV_PATH)
    .option("header", "true")
    .start()
)

In [125]:
bronze_csv_query.isActive

True

In [126]:
bronze_csv_query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': True}

In [127]:
spark.read.csv(
    BRONZE_CSV_PATH,
    header=True
).show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|raw_json                                                                                                                                                                                                                                                                  |topic             |partition|offset|kafka_timestamp         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|{"event_i

In [128]:
bronze_csv_query.isActive

True

In [129]:
bronze_df = spark.read.csv(
    BRONZE_CSV_PATH,
    header=True
)

bronze_df.show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|raw_json                                                                                                                                                                                                                                                                  |topic             |partition|offset|kafka_timestamp         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|{"event_i

In [130]:
bronze_df.count()

20000

In [131]:
bronze_df = spark.read.csv(
    BRONZE_CSV_PATH,
    header=True
)

bronze_df.show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|raw_json                                                                                                                                                                                                                                                                  |topic             |partition|offset|kafka_timestamp         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|{"event_i

In [132]:
bronze_csv_query.stop()

In [133]:
spark.streams.active

[]

In [134]:
bronze_df.printSchema()

root
 |-- raw_json: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: string (nullable = true)
 |-- offset: string (nullable = true)
 |-- kafka_timestamp: string (nullable = true)



In [135]:
bronze_df.printSchema()

root
 |-- raw_json: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: string (nullable = true)
 |-- offset: string (nullable = true)
 |-- kafka_timestamp: string (nullable = true)



In [136]:
spark.sql("""
CREATE TABLE IF NOT EXISTS retailpulse_bronze.streaming_events (
    raw_json STRING,
    topic STRING,
    partition STRING,
    offset STRING,
    kafka_timestamp STRING
)
USING CSV
OPTIONS (
    header 'true'
)
LOCATION 'hdfs://namenode:8020/data/retailpulse/bronze/streaming/events_csv'
""")

DataFrame[]

In [137]:
spark.sql("""
SHOW TABLES IN retailpulse_bronze
""").show(truncate=False)

+------------------+-------------------+-----------+
|database          |tableName          |isTemporary|
+------------------+-------------------+-----------+
|retailpulse_bronze|customers          |false      |
|retailpulse_bronze|fulfillment_events |false      |
|retailpulse_bronze|inventory_snapshots|false      |
|retailpulse_bronze|order_items        |false      |
|retailpulse_bronze|orders             |false      |
|retailpulse_bronze|payments           |false      |
|retailpulse_bronze|products           |false      |
|retailpulse_bronze|stores             |false      |
|retailpulse_bronze|streaming_events   |false      |
|                  |retailpulse_test   |true       |
+------------------+-------------------+-----------+



In [138]:
spark.sql("""
SELECT *
FROM retailpulse_bronze.streaming_events
LIMIT 10
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|raw_json                                                                                                                                                                                                                                                                  |topic             |partition|offset|kafka_timestamp         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------+------+------------------------+
|{"event_i

In [139]:
spark.sql("""
SHOW TABLES IN retailpulse_bronze
""").show(truncate=False)

+------------------+-------------------+-----------+
|database          |tableName          |isTemporary|
+------------------+-------------------+-----------+
|retailpulse_bronze|customers          |false      |
|retailpulse_bronze|fulfillment_events |false      |
|retailpulse_bronze|inventory_snapshots|false      |
|retailpulse_bronze|order_items        |false      |
|retailpulse_bronze|orders             |false      |
|retailpulse_bronze|payments           |false      |
|retailpulse_bronze|products           |false      |
|retailpulse_bronze|stores             |false      |
|retailpulse_bronze|streaming_events   |false      |
|                  |retailpulse_test   |true       |
+------------------+-------------------+-----------+



## Transformation

In [140]:
bronze_tables = [
    "customers",
    "fulfillment_events",
    "inventory_snapshots",
    "order_items",
    "orders",
    "payments",
    "products",
    "stores",
    "streaming_events"
]



silver_tables = [
    # سيتم إنشاؤها بعد الـ transformations
]

gold_tables = [
 # سيتم إنشاؤها بعد بناء الـ marts
]

In [141]:
def read_tables(tables):
    dataframes = {}

    for table in tables:
        dataframes[table] = spark.table(f"retailpulse_bronze.{table}")
        print(f"Loaded: {table}")

    return dataframes
                 
    

In [142]:
dfs=read_tables(bronze_tables)

Loaded: customers
Loaded: fulfillment_events
Loaded: inventory_snapshots
Loaded: order_items
Loaded: orders
Loaded: payments
Loaded: products
Loaded: stores
Loaded: streaming_events


In [143]:
from pyspark.sql import functions as F


def is_null_value(column):
    return (
        F.col(column).isNull() |
        (F.lower(F.trim(F.col(column).cast("string"))) == "null")
    )

In [144]:
def check_data(df_tables, primary_keys):

    for table_name, df in df_tables.items():

        print("\n" + "=" * 70)
        print(f"TABLE: {table_name}")
        print("=" * 70)

        # --------------------------------------------------
        # 1. Schema
        # --------------------------------------------------
        print("\n--- Schema ---")
        df.printSchema()

        # --------------------------------------------------
        # 2. Head
        # --------------------------------------------------
        print("\n--- Head ---")
        df.show(5, truncate=False)

        # --------------------------------------------------
        # 3. Row Count
        # --------------------------------------------------
        row_count = df.count()
        print(f"\n--- Row Count ---")
        print(row_count)

        # --------------------------------------------------
        # 4. NULL / "null" count per column
        # --------------------------------------------------
        print("\n--- NULL Count per Column ---")

        null_counts = df.select([
            F.sum(
                F.when(
                    is_null_value(column),
                    1
                ).otherwise(0)
            ).alias(column)
            for column in df.columns
        ])

        null_counts.show()

        # --------------------------------------------------
        # 5. Duplicate values per column
        # --------------------------------------------------
        print("\n--- Duplicate Values per Column ---")

        for column in df.columns:

            duplicate_count = (
                df.groupBy(column)
                  .count()
                  .filter(
                      (F.col("count") > 1) &
                      ~is_null_value(column)
                  )
                  .select(
                      F.sum(F.col("count") - 1).alias("duplicates")
                  )
                  .collect()[0]["duplicates"]
            )

            if duplicate_count is None:
                duplicate_count = 0

            print(
                f"{column}: {duplicate_count} duplicate occurrences"
            )

        # --------------------------------------------------
        # 6. Primary Key validation
        # --------------------------------------------------
        if table_name in primary_keys:

            pk = primary_keys[table_name]

            print(f"\n--- Primary Key Check: {pk} ---")

            # NULL / "null" PK
            null_pk_count = df.filter(
                is_null_value(pk)
            ).count()

            # Duplicate PK values
            duplicate_pk_count = (
                df.groupBy(pk)
                  .count()
                  .filter(
                      (F.col("count") > 1) &
                      ~is_null_value(pk)
                  )
                  .count()
            )

            print(f"NULL Primary Keys: {null_pk_count}")
            print(f"Duplicate Primary Key Values: {duplicate_pk_count}")

In [145]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "stores": "store_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id",
    "fulfillment_events": "fulfillment_event_id",
    "inventory_snapshots": "inventory_snapshot_id"
}

In [146]:
check_data(dfs, primary_keys)


TABLE: customers

--- Schema ---
root
 |-- customer_id: integer (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- signup_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)


--- Head ---
+-----------+----------+---------------------+--------------+------------+-------------------+-------------------+
|customer_id|full_name |email                |phone         |country_code|signup_at          |updated_at         |
+-----------+----------+---------------------+--------------+------------+-------------------+-------------------+
|1          |Customer 1|customer1@example.com|+20-10-0000001|SA          |2026-01-01 00:01:00|2026-02-10 12:49:00|
|2          |Customer 2|customer2@example.com|+20-10-0000002|Egypt       |2026-01-01 00:02:00|2026-03-31 03:53:00|
|3          |Customer 3|customer3@example.com|+20-10-0000003|AE          |2026-

+-------------+--------+----------+--------+----------+-------------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|line_discount|updated_at|
+-------------+--------+----------+--------+----------+-------------+----------+
|            0|       0|         0|       0|         0|            0|         0|
+-------------+--------+----------+--------+----------+-------------+----------+


--- Duplicate Values per Column ---
order_item_id: 0 duplicate occurrences
order_id: 199621 duplicate occurrences
product_id: 289621 duplicate occurrences
quantity: 299611 duplicate occurrences
unit_price: 213655 duplicate occurrences
line_discount: 267420 duplicate occurrences
updated_at: 199621 duplicate occurrences

--- Primary Key Check: order_item_id ---
NULL Primary Keys: 0
Duplicate Primary Key Values: 0

TABLE: orders

--- Schema ---
root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- order_status: 

+--------+----------+----+------------+---------+
|store_id|store_name|city|country_code|opened_at|
+--------+----------+----+------------+---------+
|       0|         0|   0|           0|        0|
+--------+----------+----+------------+---------+


--- Duplicate Values per Column ---
store_id: 0 duplicate occurrences
store_name: 0 duplicate occurrences
city: 30 duplicate occurrences
country_code: 49 duplicate occurrences
opened_at: 49 duplicate occurrences

--- Primary Key Check: store_id ---
NULL Primary Keys: 0
Duplicate Primary Key Values: 0

TABLE: streaming_events

--- Schema ---
root
 |-- raw_json: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: string (nullable = true)
 |-- offset: string (nullable = true)
 |-- kafka_timestamp: string (nullable = true)


--- Head ---
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [147]:
from pyspark.sql import functions as F

def check_primary_keys(df_tables, primary_keys):

    for table_name, pk in primary_keys.items():

        df = df_tables[table_name]

        null_count = df.filter(
            F.col(pk).isNull()
        ).count()

        duplicate_count = (
            df.groupBy(pk)
              .count()
              .filter(F.col("count") > 1)
              .count()
        )

        print(f"\n------ {table_name} ------")
        print(f"Primary Key: {pk}")
        print(f"NULL PKs: {null_count}")
        print(f"Duplicate PKs: {duplicate_count}")

In [148]:
check_primary_keys(dfs, primary_keys)


------ customers ------
Primary Key: customer_id
NULL PKs: 0
Duplicate PKs: 0

------ products ------
Primary Key: product_id
NULL PKs: 0
Duplicate PKs: 0

------ stores ------
Primary Key: store_id
NULL PKs: 0
Duplicate PKs: 0

------ orders ------
Primary Key: order_id
NULL PKs: 0
Duplicate PKs: 0

------ order_items ------
Primary Key: order_item_id
NULL PKs: 0
Duplicate PKs: 0

------ payments ------
Primary Key: payment_id
NULL PKs: 0
Duplicate PKs: 0

------ fulfillment_events ------
Primary Key: fulfillment_event_id
NULL PKs: 0
Duplicate PKs: 0

------ inventory_snapshots ------
Primary Key: inventory_snapshot_id
NULL PKs: 0
Duplicate PKs: 0


In [149]:
from pyspark.sql import functions as F


# ============================================================
# CUSTOMERS - DEEP CLEANING
# Bronze → Silver
# ============================================================

customers_df = dfs["customers"]


# ------------------------------------------------------------
# 1. Select required columns
# ------------------------------------------------------------

customers_silver_df = customers_df.select(
    "customer_id",
    "full_name",
    "email",
    "phone",
    "country_code",
    "signup_at",
    "updated_at"
)


# ------------------------------------------------------------
# 2. Clean string columns
#    - trim spaces
#    - "null" → NULL
#    - empty string → NULL
# ------------------------------------------------------------

string_columns = [
    "full_name",
    "email",
    "phone",
    "country_code"
]

for column in string_columns:

    customers_silver_df = customers_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column).cast("string"))) == "null") |
            (F.trim(F.col(column).cast("string")) == ""),
            None
        ).otherwise(
            F.trim(F.col(column).cast("string"))
        )
    )


# ------------------------------------------------------------
# 3. Standardize full_name
#    Ahmed ALI → Ahmed Ali
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "full_name",
    F.initcap(F.lower(F.col("full_name")))
)


# ------------------------------------------------------------
# 4. Standardize email
#    Ahmed@GMAIL.COM → ahmed@gmail.com
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "email",
    F.lower(F.trim(F.col("email")))
)


# ------------------------------------------------------------
# 5. Standardize phone
#    Remove:
#    spaces
#    -
#    (
#    )
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "phone",
    F.regexp_replace(
        F.col("phone"),
        r"[\s\-\(\)]",
        ""
    )
)


# ------------------------------------------------------------
# 6. Standardize country_code
#
#    Egypt / egypt / EGY / eg → EG
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "country_code",
    F.lower(F.trim(F.col("country_code")))
)

customers_silver_df = customers_silver_df.withColumn(
    "country_code",
    F.when(
        F.col("country_code").isin(
            "egypt",
            "egy",
            "eg"
        ),
        "EG"
    ).otherwise(
        F.upper(F.col("country_code"))
    )
)


# ------------------------------------------------------------
# 7. Cast customer_id
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "customer_id",
    F.col("customer_id").cast("integer")
)


# ------------------------------------------------------------
# 8. Convert timestamps
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "signup_at",
    F.to_timestamp("signup_at")
)

customers_silver_df = customers_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# 9. Remove invalid email values
#
#    Any email that doesn't match the expected format
#    becomes NULL
# ------------------------------------------------------------

customers_silver_df = customers_silver_df.withColumn(
    "email",
    F.when(
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        F.col("email")
    ).otherwise(None)
)


# ------------------------------------------------------------
# 10. Remove records with NULL values
#
#     After all cleaning operations,
#     drop rows containing ANY NULL.
# ------------------------------------------------------------

before_drop = customers_silver_df.count()

customers_silver_df = customers_silver_df.dropna(
    how="any"
)

after_drop = customers_silver_df.count()


# ------------------------------------------------------------
# 11. Remove duplicate customers
#
#     customer_id is the Primary Key.
#     Keep one record per customer_id.
# ------------------------------------------------------------

before_duplicates = customers_silver_df.count()

customers_silver_df = customers_silver_df.dropDuplicates(
    ["customer_id"]
)

after_duplicates = customers_silver_df.count()


# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

print("=" * 70)
print("CUSTOMERS DEEP CLEANING RESULT")
print("=" * 70)

print(f"Rows before NULL removal      : {before_drop}")
print(f"Rows after NULL removal       : {after_drop}")
print(f"NULL rows removed             : {before_drop - after_drop}")

print()

print(f"Rows before duplicate removal : {before_duplicates}")
print(f"Rows after duplicate removal  : {after_duplicates}")
print(f"Duplicate rows removed        : {before_duplicates - after_duplicates}")

print()

print("Final Schema:")
customers_silver_df.printSchema()

print("Final Data:")
customers_silver_df.show(
    10,
    truncate=False
)


# ------------------------------------------------------------
# 13. Final NULL check
# ------------------------------------------------------------

print("Final NULL Check:")

customers_silver_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in customers_silver_df.columns
]).show()

CUSTOMERS DEEP CLEANING RESULT
Rows before NULL removal      : 40000
Rows after NULL removal       : 39415
NULL rows removed             : 585

Rows before duplicate removal : 39415
Rows after duplicate removal  : 39415
Duplicate rows removed        : 0

Final Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- signup_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

Final Data:
+-----------+-------------+------------------------+------------+------------+-------------------+-------------------+
|customer_id|full_name    |email                   |phone       |country_code|signup_at          |updated_at         |
+-----------+-------------+------------------------+------------+------------+-------------------+-------------------+
|148        |Customer 148 |customer148@example.com |+20100000148

In [150]:
print(f"({customers_silver_df.count()}, {len(customers_silver_df.columns)})")

(39415, 7)


In [151]:
from pyspark.sql import functions as F


# ============================================================
# PRODUCTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

products_df = dfs["products"]


# ------------------------------------------------------------
# 1. Select required columns
# ------------------------------------------------------------

products_silver_df = products_df.select(
    "product_id",
    "sku",
    "product_name",
    "category",
    "unit_cost",
    "list_price",
    "updated_at"
)


# ------------------------------------------------------------
# 2. Clean string columns
#    - trim spaces
#    - "null" → NULL
#    - empty string → NULL
# ------------------------------------------------------------

string_columns = [
    "sku",
    "product_name",
    "category"
]

for column in string_columns:

    products_silver_df = products_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column).cast("string"))) == "null") |
            (F.trim(F.col(column).cast("string")) == ""),
            None
        ).otherwise(
            F.trim(F.col(column).cast("string"))
        )
    )


# ------------------------------------------------------------
# 3. Standardize SKU
#    abc-001 → ABC-001
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "sku",
    F.upper(F.col("sku"))
)


# ------------------------------------------------------------
# 4. Standardize Product Name
#    iphone 15 → Iphone 15
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "product_name",
    F.initcap(F.lower(F.col("product_name")))
)


# ------------------------------------------------------------
# 5. Standardize Category
#    electronics → Electronics
#    ELECTRONICS → Electronics
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "category",
    F.initcap(F.lower(F.col("category")))
)


# ------------------------------------------------------------
# 6. Cast IDs
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "product_id",
    F.col("product_id").cast("integer")
)


# ------------------------------------------------------------
# 7. Cast financial columns
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "unit_cost",
    F.col("unit_cost").cast("decimal(18,2)")
)

products_silver_df = products_silver_df.withColumn(
    "list_price",
    F.col("list_price").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# 8. Convert timestamp
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# 9. Handle invalid negative values
#
#    Negative cost/price is invalid for this product dataset.
#    Convert to NULL first.
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "unit_cost",
    F.when(
        F.col("unit_cost") < 0,
        None
    ).otherwise(
        F.col("unit_cost")
    )
)

products_silver_df = products_silver_df.withColumn(
    "list_price",
    F.when(
        F.col("list_price") < 0,
        None
    ).otherwise(
        F.col("list_price")
    )
)


# ------------------------------------------------------------
# 10. Business Rule
#
#     A product's cost should not be greater than
#     its selling/list price.
#
#     unit_cost > list_price → invalid
# ------------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "invalid_price",
    F.col("unit_cost") > F.col("list_price")
)

products_silver_df = products_silver_df.withColumn(
    "unit_cost",
    F.when(
        F.col("invalid_price"),
        None
    ).otherwise(
        F.col("unit_cost")
    )
)

products_silver_df = products_silver_df.withColumn(
    "list_price",
    F.when(
        F.col("invalid_price"),
        None
    ).otherwise(
        F.col("list_price")
    )
)


# ------------------------------------------------------------
# 11. Remove temporary validation column
# ------------------------------------------------------------

products_silver_df = products_silver_df.drop(
    "invalid_price"
)


# ------------------------------------------------------------
# 12. Remove rows containing NULL
# ------------------------------------------------------------

before_null_drop = products_silver_df.count()

products_silver_df = products_silver_df.dropna(
    how="any"
)

after_null_drop = products_silver_df.count()


# ------------------------------------------------------------
# 13. Remove duplicate product_id
# ------------------------------------------------------------

before_product_duplicates = products_silver_df.count()

products_silver_df = products_silver_df.dropDuplicates(
    ["product_id"]
)

after_product_duplicates = products_silver_df.count()


# ------------------------------------------------------------
# 14. Remove duplicate SKU
# ------------------------------------------------------------

before_sku_duplicates = products_silver_df.count()

products_silver_df = products_silver_df.dropDuplicates(
    ["sku"]
)

after_sku_duplicates = products_silver_df.count()


# ------------------------------------------------------------
# 15. Final result
# ------------------------------------------------------------

print("=" * 70)
print("PRODUCTS DEEP CLEANING RESULT")
print("=" * 70)

print(f"Rows before NULL removal       : {before_null_drop}")
print(f"Rows after NULL removal        : {after_null_drop}")
print(f"NULL rows removed              : {before_null_drop - after_null_drop}")

print()

print(f"Rows before product duplicates : {before_product_duplicates}")
print(f"Rows after product duplicates  : {after_product_duplicates}")
print(
    f"Duplicate product_id removed   : "
    f"{before_product_duplicates - after_product_duplicates}"
)

print()

print(f"Rows before SKU duplicates     : {before_sku_duplicates}")
print(f"Rows after SKU duplicates      : {after_sku_duplicates}")
print(
    f"Duplicate SKU removed          : "
    f"{before_sku_duplicates - after_sku_duplicates}"
)

print()

print("Final Schema:")
products_silver_df.printSchema()

print("Final Data:")
products_silver_df.show(
    10,
    truncate=False
)


# ------------------------------------------------------------
# 16. Final NULL check
# ------------------------------------------------------------

print("Final NULL Check:")

products_silver_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in products_silver_df.columns
]).show()

PRODUCTS DEEP CLEANING RESULT
Rows before NULL removal       : 10000
Rows after NULL removal        : 10000
NULL rows removed              : 0

Rows before product duplicates : 10000
Rows after product duplicates  : 10000
Duplicate product_id removed   : 0

Rows before SKU duplicates     : 10000
Rows after SKU duplicates      : 10000
Duplicate SKU removed          : 0

Final Schema:
root
 |-- product_id: integer (nullable = true)
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_cost: decimal(18,2) (nullable = true)
 |-- list_price: decimal(18,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

Final Data:
+----------+----------+------------+-----------+---------+----------+-------------------+
|product_id|sku       |product_name|category   |unit_cost|list_price|updated_at         |
+----------+----------+------------+-----------+---------+----------+-------------------+
|89        |SKU-000089

In [152]:
from pyspark.sql import functions as F


# ============================================================
# STORES - DEEP CLEANING
# Bronze → Silver
# ============================================================

stores_df = dfs["stores"]


stores_silver_df = stores_df.select(
    "store_id",
    "store_name",
    "city",
    "country_code",
    "opened_at"
)


# ------------------------------------------------------------
# Clean strings
# ------------------------------------------------------------

for column in ["store_name", "city", "country_code"]:

    stores_silver_df = stores_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column))) == "null") |
            (F.trim(F.col(column)) == ""),
            None
        ).otherwise(
            F.trim(F.col(column))
        )
    )


# ------------------------------------------------------------
# Standardize store name and city
# ------------------------------------------------------------

stores_silver_df = stores_silver_df.withColumn(
    "store_name",
    F.initcap(F.lower(F.col("store_name")))
)

stores_silver_df = stores_silver_df.withColumn(
    "city",
    F.initcap(F.lower(F.col("city")))
)


# ------------------------------------------------------------
# Standardize country
# ------------------------------------------------------------

stores_silver_df = stores_silver_df.withColumn(
    "country_code",
    F.lower(F.trim(F.col("country_code")))
)

stores_silver_df = stores_silver_df.withColumn(
    "country_code",
    F.when(
        F.col("country_code").isin("egypt", "egy", "eg"),
        "EG"
    ).otherwise(
        F.upper(F.col("country_code"))
    )
)


# ------------------------------------------------------------
# Cast ID and date
# ------------------------------------------------------------

stores_silver_df = stores_silver_df.withColumn(
    "store_id",
    F.col("store_id").cast("integer")
)

stores_silver_df = stores_silver_df.withColumn(
    "opened_at",
    F.to_date("opened_at")
)


# ------------------------------------------------------------
# Remove NULLs
# ------------------------------------------------------------

before_null = stores_silver_df.count()

stores_silver_df = stores_silver_df.dropna(
    how="any"
)

after_null = stores_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate store_id
# ------------------------------------------------------------

before_dup = stores_silver_df.count()

stores_silver_df = stores_silver_df.dropDuplicates(
    ["store_id"]
)

after_dup = stores_silver_df.count()


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("========== STORES SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

stores_silver_df.printSchema()
stores_silver_df.show(10, truncate=False)

========== STORES SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- opened_at: date (nullable = true)

+--------+---------------------+-------+------------+----------+
|store_id|store_name           |city   |country_code|opened_at |
+--------+---------------------+-------+------------+----------+
|31      |Retailpulse Store 031|City 11|EG          |2026-01-01|
|34      |Retailpulse Store 034|City 14|EG          |2026-01-01|
|28      |Retailpulse Store 028|City 08|EG          |2026-01-01|
|26      |Retailpulse Store 026|City 06|EG          |2026-01-01|
|27      |Retailpulse Store 027|City 07|EG          |2026-01-01|
|44      |Retailpulse Store 044|City 04|EG          |2026-01-01|
|12      |Retailpulse Store 012|City 12|EG          |2026-01-01|
|22      |Retailpulse Store 022|City 02|EG          

In [153]:
from pyspark.sql import functions as F


# ============================================================
# ORDER_ITEMS - DEEP CLEANING
# Bronze → Silver
# ============================================================

order_items_df = dfs["order_items"]


order_items_silver_df = order_items_df.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_discount",
    "updated_at"
)


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

for column in ["order_item_id", "order_id"]:

    order_items_silver_df = order_items_silver_df.withColumn(
        column,
        F.col(column).cast("long")
    )


order_items_silver_df = order_items_silver_df.withColumn(
    "product_id",
    F.col("product_id").cast("integer")
)


# ------------------------------------------------------------
# Cast numeric values
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "quantity",
    F.col("quantity").cast("integer")
)

order_items_silver_df = order_items_silver_df.withColumn(
    "unit_price",
    F.col("unit_price").cast("decimal(18,2)")
)

order_items_silver_df = order_items_silver_df.withColumn(
    "line_discount",
    F.col("line_discount").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# Timestamp
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# Business rules
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "quantity",
    F.when(
        F.col("quantity") <= 0,
        None
    ).otherwise(F.col("quantity"))
)

order_items_silver_df = order_items_silver_df.withColumn(
    "unit_price",
    F.when(
        F.col("unit_price") < 0,
        None
    ).otherwise(F.col("unit_price"))
)

order_items_silver_df = order_items_silver_df.withColumn(
    "line_discount",
    F.when(
        F.col("line_discount") < 0,
        None
    ).otherwise(F.col("line_discount"))
)


# ------------------------------------------------------------
# Discount cannot exceed line amount
# quantity × unit_price
# ------------------------------------------------------------

order_items_silver_df = order_items_silver_df.withColumn(
    "line_discount",
    F.when(
        F.col("line_discount") >
        (F.col("quantity") * F.col("unit_price")),
        None
    ).otherwise(
        F.col("line_discount")
    )
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

for column in [
    "order_item_id",
    "order_id",
    "product_id"
]:

    order_items_silver_df = order_items_silver_df.withColumn(
        column,
        F.when(
            F.col(column) <= 0,
            None
        ).otherwise(F.col(column))
    )


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = order_items_silver_df.count()

order_items_silver_df = order_items_silver_df.dropna(
    how="any"
)

after_null = order_items_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate order_item_id
# ------------------------------------------------------------

before_dup = order_items_silver_df.count()

order_items_silver_df = order_items_silver_df.dropDuplicates(
    ["order_item_id"]
)

after_dup = order_items_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== ORDER_ITEMS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

order_items_silver_df.printSchema()
order_items_silver_df.show(10, truncate=False)

========== ORDER_ITEMS SILVER ==========
NULL rows removed      : 866
Duplicate rows removed : 0
root
 |-- order_item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- line_discount: decimal(18,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+-------------+--------+----------+--------+----------+-------------+-------------------+
|order_item_id|order_id|product_id|quantity|unit_price|line_discount|updated_at         |
+-------------+--------+----------+--------+----------+-------------+-------------------+
|26           |7       |8168      |2       |558.94    |111.79       |2026-01-01 00:28:00|
|29           |8       |9353      |4       |397.16    |0.00         |2026-01-01 00:32:00|
|474          |154     |67        |3       |485.19    |0.00         |2026-01-01 10:16:00|
|964          |320     |8434      |2       |29

In [154]:
from pyspark.sql import functions as F


# ============================================================
# PAYMENTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

payments_df = dfs["payments"]


payments_silver_df = payments_df.select(
    "payment_id",
    "order_id",
    "payment_method",
    "payment_status",
    "amount",
    "paid_at",
    "updated_at"
)


# ------------------------------------------------------------
# Clean strings
# ------------------------------------------------------------

for column in ["payment_method", "payment_status"]:

    payments_silver_df = payments_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column))) == "null") |
            (F.trim(F.col(column)) == ""),
            None
        ).otherwise(
            F.lower(F.trim(F.col(column)))
        )
    )


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "payment_id",
    F.col("payment_id").cast("long")
)

payments_silver_df = payments_silver_df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)


# ------------------------------------------------------------
# Cast amount
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "amount",
    F.col("amount").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# Timestamps
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "paid_at",
    F.to_timestamp("paid_at")
)

payments_silver_df = payments_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# Amount cannot be negative
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "amount",
    F.when(
        F.col("amount") < 0,
        None
    ).otherwise(F.col("amount"))
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

payments_silver_df = payments_silver_df.withColumn(
    "payment_id",
    F.when(
        F.col("payment_id") <= 0,
        None
    ).otherwise(F.col("payment_id"))
)

payments_silver_df = payments_silver_df.withColumn(
    "order_id",
    F.when(
        F.col("order_id") <= 0,
        None
    ).otherwise(F.col("order_id"))
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = payments_silver_df.count()

payments_silver_df = payments_silver_df.dropna(
    how="any"
)

after_null = payments_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate payment_id
# ------------------------------------------------------------

before_dup = payments_silver_df.count()

payments_silver_df = payments_silver_df.dropDuplicates(
    ["payment_id"]
)

after_dup = payments_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== PAYMENTS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

payments_silver_df.printSchema()
payments_silver_df.show(10, truncate=False)

========== PAYMENTS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- paid_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+----------+--------+--------------+--------------+-------+-------------------+-------------------+
|payment_id|order_id|payment_method|payment_status|amount |paid_at            |updated_at         |
+----------+--------+--------------+--------------+-------+-------------------+-------------------+
|26        |26      |cash          |captured      |3911.78|2026-01-01 01:46:00|2026-01-01 01:44:00|
|29        |29      |card          |captured      |4638.17|2026-01-01 01:58:00|2026-01-01 01:56:00|
|474       |474     |card          |failed        |3845.55|2026-01-02 07:38:00|2026-01-02 07:36:00|


In [155]:
from pyspark.sql import functions as F


# ============================================================
# FULFILLMENT_EVENTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

fulfillment_df = dfs["fulfillment_events"]


fulfillment_silver_df = fulfillment_df.select(
    "fulfillment_event_id",
    "order_id",
    "event_type",
    "event_timestamp",
    "warehouse_code",
    "updated_at"
)


# ------------------------------------------------------------
# Clean strings
# ------------------------------------------------------------

for column in ["event_type", "warehouse_code"]:

    fulfillment_silver_df = fulfillment_silver_df.withColumn(
        column,
        F.when(
            F.col(column).isNull() |
            (F.lower(F.trim(F.col(column))) == "null") |
            (F.trim(F.col(column)) == ""),
            None
        ).otherwise(
            F.lower(F.trim(F.col(column)))
        )
    )


# ------------------------------------------------------------
# IDs
# ------------------------------------------------------------

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "fulfillment_event_id",
    F.col("fulfillment_event_id").cast("long")
)

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)


# ------------------------------------------------------------
# Timestamps
# ------------------------------------------------------------

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "event_timestamp",
    F.to_timestamp("event_timestamp")
)

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "fulfillment_event_id",
    F.when(
        F.col("fulfillment_event_id") <= 0,
        None
    ).otherwise(F.col("fulfillment_event_id"))
)

fulfillment_silver_df = fulfillment_silver_df.withColumn(
    "order_id",
    F.when(
        F.col("order_id") <= 0,
        None
    ).otherwise(F.col("order_id"))
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = fulfillment_silver_df.count()

fulfillment_silver_df = fulfillment_silver_df.dropna(
    how="any"
)

after_null = fulfillment_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate event ID
# ------------------------------------------------------------

before_dup = fulfillment_silver_df.count()

fulfillment_silver_df = fulfillment_silver_df.dropDuplicates(
    ["fulfillment_event_id"]
)

after_dup = fulfillment_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== FULFILLMENT_EVENTS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

fulfillment_silver_df.printSchema()
fulfillment_silver_df.show(10, truncate=False)

========== FULFILLMENT_EVENTS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- fulfillment_event_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- warehouse_code: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+--------------------+--------+----------+-------------------+--------------+-------------------+
|fulfillment_event_id|order_id|event_type|event_timestamp    |warehouse_code|updated_at         |
+--------------------+--------+----------+-------------------+--------------+-------------------+
|26                  |9       |shipped   |2026-01-01 12:36:00|wh-5          |2026-01-01 12:36:00|
|29                  |10      |shipped   |2026-01-01 12:40:00|wh-1          |2026-01-01 12:40:00|
|474                 |158     |delivered |2026-01-03 22:32:00|wh-4          |2026-01-03 22:32:00|
|964                 |322     |pa

In [156]:
from pyspark.sql import functions as F


# ============================================================
# ORDERS - DEEP CLEANING
# Bronze → Silver
# ============================================================

orders_df = dfs["orders"]


orders_silver_df = orders_df.select(
    "order_id",
    "customer_id",
    "store_id",
    "order_status",
    "order_timestamp",
    "order_total",
    "discount_amount",
    "updated_at"
)
# ------------------------------------------------------------
# Standardize order_status
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    F.lower(
        F.trim(
            F.col("order_status")
        )
    )
)

orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    F.when(
        F.col("order_status").isin(
            "completed",
            "compeleted",
            "complete",
            "complated"
        ),
        "completed"
    )
    .when(
        F.col("order_status").isin(
            "cancelled",
            "canceled",
            "cancel"
        ),
        "cancelled"
    )
    .when(
        F.col("order_status").isin(
            "pending",
            "pendding"
        ),
        "pending"
    )
    .when(
        F.col("order_status").isin(
            "processing",
            "process"
        ),
        "processing"
    )
    .otherwise(
        F.col("order_status")
    )
)

# ------------------------------------------------------------
# Clean order_status
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    F.when(
        F.col("order_status").isNull() |
        (F.lower(F.trim(F.col("order_status"))) == "null") |
        (F.trim(F.col("order_status")) == ""),
        None
    ).otherwise(
        F.lower(F.trim(F.col("order_status")))
    )
)


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_id",
    F.col("order_id").cast("long")
)

orders_silver_df = orders_silver_df.withColumn(
    "customer_id",
    F.col("customer_id").cast("integer")
)

orders_silver_df = orders_silver_df.withColumn(
    "store_id",
    F.col("store_id").cast("integer")
)


# ------------------------------------------------------------
# Cast financial columns
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_total",
    F.col("order_total").cast("decimal(18,2)")
)

orders_silver_df = orders_silver_df.withColumn(
    "discount_amount",
    F.col("discount_amount").cast("decimal(18,2)")
)


# ------------------------------------------------------------
# Convert timestamps
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_timestamp",
    F.to_timestamp("order_timestamp")
)

orders_silver_df = orders_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# Negative financial values → NULL
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_total",
    F.when(
        F.col("order_total") < 0,
        None
    ).otherwise(F.col("order_total"))
)

orders_silver_df = orders_silver_df.withColumn(
    "discount_amount",
    F.when(
        F.col("discount_amount") < 0,
        None
    ).otherwise(F.col("discount_amount"))
)


# ------------------------------------------------------------
# Business rule:
# discount cannot exceed order total
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "discount_amount",
    F.when(
        F.col("discount_amount") > F.col("order_total"),
        None
    ).otherwise(
        F.col("discount_amount")
    )
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

orders_silver_df = orders_silver_df.withColumn(
    "order_id",
    F.when(F.col("order_id") <= 0, None)
    .otherwise(F.col("order_id"))
)

orders_silver_df = orders_silver_df.withColumn(
    "customer_id",
    F.when(F.col("customer_id") <= 0, None)
    .otherwise(F.col("customer_id"))
)

orders_silver_df = orders_silver_df.withColumn(
    "store_id",
    F.when(F.col("store_id") <= 0, None)
    .otherwise(F.col("store_id"))
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = orders_silver_df.count()

orders_silver_df = orders_silver_df.dropna(
    how="any"
)

after_null = orders_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate order_id
# ------------------------------------------------------------

before_dup = orders_silver_df.count()

orders_silver_df = orders_silver_df.dropDuplicates(
    ["order_id"]
)

after_dup = orders_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== ORDERS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

orders_silver_df.printSchema()
orders_silver_df.show(10, truncate=False)

========== ORDERS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- order_total: decimal(18,2) (nullable = true)
 |-- discount_amount: decimal(18,2) (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+--------+-----------+--------+------------+-------------------+-----------+---------------+-------------------+
|order_id|customer_id|store_id|order_status|order_timestamp    |order_total|discount_amount|updated_at         |
+--------+-----------+--------+------------+-------------------+-----------+---------------+-------------------+
|26      |23657      |46      |cancelled   |2026-01-01 01:44:00|3911.78    |0.00           |2026-01-01 02:43:00|
|29      |26746      |21      |completed   |2026-01-01 01:56:00|4638.17    |0.00      

In [157]:
from pyspark.sql import functions as F


# ============================================================
# INVENTORY_SNAPSHOTS - DEEP CLEANING
# Bronze → Silver
# ============================================================

inventory_df = dfs["inventory_snapshots"]


inventory_silver_df = inventory_df.select(
    "inventory_snapshot_id",
    "product_id",
    "store_id",
    "stock_on_hand",
    "snapshot_at",
    "updated_at"
)


# ------------------------------------------------------------
# Cast IDs
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "inventory_snapshot_id",
    F.col("inventory_snapshot_id").cast("long")
)

inventory_silver_df = inventory_silver_df.withColumn(
    "product_id",
    F.col("product_id").cast("integer")
)

inventory_silver_df = inventory_silver_df.withColumn(
    "store_id",
    F.col("store_id").cast("integer")
)


# ------------------------------------------------------------
# Cast stock
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "stock_on_hand",
    F.col("stock_on_hand").cast("integer")
)


# ------------------------------------------------------------
# Timestamps
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "snapshot_at",
    F.to_timestamp("snapshot_at")
)

inventory_silver_df = inventory_silver_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)


# ------------------------------------------------------------
# IDs must be positive
# ------------------------------------------------------------

for column in [
    "inventory_snapshot_id",
    "product_id",
    "store_id"
]:

    inventory_silver_df = inventory_silver_df.withColumn(
        column,
        F.when(
            F.col(column) <= 0,
            None
        ).otherwise(F.col(column))
    )


# ------------------------------------------------------------
# Stock cannot be negative
# ------------------------------------------------------------

inventory_silver_df = inventory_silver_df.withColumn(
    "stock_on_hand",
    F.when(
        F.col("stock_on_hand") < 0,
        None
    ).otherwise(
        F.col("stock_on_hand")
    )
)


# ------------------------------------------------------------
# Drop NULL rows
# ------------------------------------------------------------

before_null = inventory_silver_df.count()

inventory_silver_df = inventory_silver_df.dropna(
    how="any"
)

after_null = inventory_silver_df.count()


# ------------------------------------------------------------
# Remove duplicate snapshot ID
# ------------------------------------------------------------

before_dup = inventory_silver_df.count()

inventory_silver_df = inventory_silver_df.dropDuplicates(
    ["inventory_snapshot_id"]
)

after_dup = inventory_silver_df.count()


# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("========== INVENTORY_SNAPSHOTS SILVER ==========")
print(f"NULL rows removed      : {before_null - after_null}")
print(f"Duplicate rows removed : {before_dup - after_dup}")

inventory_silver_df.printSchema()
inventory_silver_df.show(10, truncate=False)

========== INVENTORY_SNAPSHOTS SILVER ==========
NULL rows removed      : 0
Duplicate rows removed : 0
root
 |-- inventory_snapshot_id: long (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- snapshot_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+---------------------+----------+--------+-------------+-------------------+-------------------+
|inventory_snapshot_id|product_id|store_id|stock_on_hand|snapshot_at        |updated_at         |
+---------------------+----------+--------+-------------+-------------------+-------------------+
|26                   |3         |4       |109          |2026-01-06 00:00:00|2026-01-06 00:00:00|
|29                   |3         |4       |120          |2026-01-09 00:00:00|2026-01-09 00:00:00|
|474                  |48        |49      |70           |2026-01-04 00:00:00|2026-01-04 00:00:00|
|964                  |97      

In [158]:
from pyspark.sql import functions as F


# ============================================================
# 1. CREATE SILVER DATABASE
# ============================================================

spark.sql("""
CREATE DATABASE IF NOT EXISTS retailpulse_silver
""")


# ============================================================
# 2. SILVER HDFS BASE PATH
# ============================================================

SILVER_BASE_PATH = (
    "hdfs://namenode:8020/"
    "data/retailpulse/silver"
)


# ============================================================
# 3. ALL CLEANED DATAFRAMES
# ============================================================

silver_dfs = {
    "customers": customers_silver_df,
    "products": products_silver_df,
    "stores": stores_silver_df,
    "orders": orders_silver_df,
    "order_items": order_items_silver_df,
    "payments": payments_silver_df,
    "fulfillment_events": fulfillment_silver_df,
    "inventory_snapshots": inventory_silver_df
}


# ============================================================
# 4. WRITE CSV FILES TO HDFS
# ============================================================

for table_name, df in silver_dfs.items():

    path = f"{SILVER_BASE_PATH}/{table_name}"

    print(f"Saving {table_name} → {path}")

    (
        df.write
        .mode("overwrite")
        .format("csv")
        .option("header", "true")
        .save(path)
    )

    print(f"✓ {table_name} saved successfully")


print("\n" + "=" * 70)
print("ALL SILVER CSV FILES SAVED TO HDFS")
print("=" * 70)

Saving customers → hdfs://namenode:8020/data/retailpulse/silver/customers
✓ customers saved successfully
Saving products → hdfs://namenode:8020/data/retailpulse/silver/products
✓ products saved successfully
Saving stores → hdfs://namenode:8020/data/retailpulse/silver/stores
✓ stores saved successfully
Saving orders → hdfs://namenode:8020/data/retailpulse/silver/orders
✓ orders saved successfully
Saving order_items → hdfs://namenode:8020/data/retailpulse/silver/order_items
✓ order_items saved successfully
Saving payments → hdfs://namenode:8020/data/retailpulse/silver/payments
✓ payments saved successfully
Saving fulfillment_events → hdfs://namenode:8020/data/retailpulse/silver/fulfillment_events
✓ fulfillment_events saved successfully
Saving inventory_snapshots → hdfs://namenode:8020/data/retailpulse/silver/inventory_snapshots
✓ inventory_snapshots saved successfully

ALL SILVER CSV FILES SAVED TO HDFS


In [159]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================
# Spark Session
# =========================================================

spark = (
    SparkSession.builder
    .appName("RetailPulse-Gold-Star-Schema")
    .master("local[*]")
    .getOrCreate()
)


# =========================================================
# Configuration
# =========================================================

SILVER_BASE_PATH = (
    "hdfs://namenode:8020/data/retailpulse/silver"
)

GOLD_BASE_PATH = (
    "hdfs://namenode:8020/data/retailpulse/gold"
)

GOLD_DATABASE = "retailpulse_gold"


print("Spark Version:", spark.version)
print("Silver Path:", SILVER_BASE_PATH)
print("Gold Path:", GOLD_BASE_PATH)
print("Gold Database:", GOLD_DATABASE)

Spark Version: 2.4.1
Silver Path: hdfs://namenode:8020/data/retailpulse/silver
Gold Path: hdfs://namenode:8020/data/retailpulse/gold
Gold Database: retailpulse_gold


In [160]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {GOLD_DATABASE}
""")

print(f"Database ready: {GOLD_DATABASE}")

Database ready: retailpulse_gold


In [161]:
def read_csv_table(table_name):
    path = f"{SILVER_BASE_PATH}/{table_name}"

    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )


stores_df = read_csv_table("stores")
customers_df = read_csv_table("customers")
products_df = read_csv_table("products")
orders_df = read_csv_table("orders")
order_items_df = read_csv_table("order_items")
payments_df = read_csv_table("payments")
fulfillment_events_df = read_csv_table("fulfillment_events")
inventory_snapshots_df = read_csv_table("inventory_snapshots")


print("All Silver CSV files loaded successfully.")

All Silver CSV files loaded successfully.


In [162]:
source_tables = {
    "stores": stores_df,
    "customers": customers_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "fulfillment_events": fulfillment_events_df,
    "inventory_snapshots": inventory_snapshots_df
}

for table_name, df in source_tables.items():
    print(f"{table_name:25} -> {df.count():,} rows")

stores                    -> 50 rows
customers                 -> 39,415 rows
products                  -> 10,000 rows
orders                    -> 100,000 rows
order_items               -> 298,755 rows
payments                  -> 100,000 rows
fulfillment_events        -> 300,000 rows
inventory_snapshots       -> 100,000 rows


In [163]:
date_range = (
    orders_df
    .select(
        F.to_date("order_timestamp").alias("order_date")
    )
    .filter(F.col("order_date").isNotNull())
    .agg(
        F.min("order_date").alias("min_date"),
        F.max("order_date").alias("max_date")
    )
    .select(
        F.date_sub("min_date", 1).alias("start_date"),
        F.date_add("max_date", 1).alias("end_date")
    )
)


dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("start_date"),
                F.col("end_date"),
                F.expr("interval 1 day")
            )
        ).alias("full_date")
    )
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin(1, 7)
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "day",
        "day_name",
        "is_weekend"
    )
    .orderBy("full_date")
)

dim_date.show(10, truncate=False)

+--------+----------+----+-------+-----+---+---------+----------+
|date_key|full_date |year|quarter|month|day|day_name |is_weekend|
+--------+----------+----+-------+-----+---+---------+----------+
|20251231|2025-12-31|2025|4      |12   |31 |Wednesday|false     |
|20260101|2026-01-01|2026|1      |1    |1  |Thursday |false     |
|20260102|2026-01-02|2026|1      |1    |2  |Friday   |false     |
|20260103|2026-01-03|2026|1      |1    |3  |Saturday |true      |
|20260104|2026-01-04|2026|1      |1    |4  |Sunday   |true      |
|20260105|2026-01-05|2026|1      |1    |5  |Monday   |false     |
|20260106|2026-01-06|2026|1      |1    |6  |Tuesday  |false     |
|20260107|2026-01-07|2026|1      |1    |7  |Wednesday|false     |
|20260108|2026-01-08|2026|1      |1    |8  |Thursday |false     |
|20260109|2026-01-09|2026|1      |1    |9  |Friday   |false     |
+--------+----------+----+-------+-----+---+---------+----------+
only showing top 10 rows



In [164]:
# =========================================================
# Save dim_date as CSV
# =========================================================

table_name = "dim_date"
path = f"{GOLD_BASE_PATH}/{table_name}"


(
    dim_date.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)


# =========================================================
# Create Hive External Table
# Compatible with Spark 2.4.1
# =========================================================

spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")


# Build Hive column definition from DataFrame schema
columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_date.schema.fields
    ]
)


spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")


print(f"✓ CSV files saved to: {path}")
print(f"✓ Hive table created: {GOLD_DATABASE}.{table_name}")

✓ CSV files saved to: hdfs://namenode:8020/data/retailpulse/gold/dim_date
✓ Hive table created: retailpulse_gold.dim_date


In [165]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_at").asc()
    )
)


customer_history = (
    customers_df
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "signup_date",
        F.to_date("signup_at")
    )
)


# Previous version
customer_history = (
    customer_history
    .withColumn(
        "previous_full_name",
        F.lag("full_name").over(customer_window)
    )
    .withColumn(
        "previous_email",
        F.lag("email").over(customer_window)
    )
    .withColumn(
        "previous_country_code",
        F.lag("country_code").over(customer_window)
    )
)


# Detect changed versions
customer_history = (
    customer_history
    .withColumn(
        "is_first_version",
        F.col("previous_full_name").isNull()
        & F.col("previous_email").isNull()
        & F.col("previous_country_code").isNull()
    )
    .withColumn(
        "has_changed",
        (
            ~F.col("full_name").eqNullSafe(
                F.col("previous_full_name")
            )
            |
            ~F.col("email").eqNullSafe(
                F.col("previous_email")
            )
            |
            ~F.col("country_code").eqNullSafe(
                F.col("previous_country_code")
            )
        )
    )
    .filter(
        F.col("is_first_version")
        | F.col("has_changed")
    )
)


# Validity
customer_scd_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("updated_at")
)


customer_history = (
    customer_history
    .withColumn(
        "scd_valid_from",
        F.col("updated_at")
    )
    .withColumn(
        "scd_valid_to",
        F.lead("updated_at").over(customer_scd_window)
    )
    .withColumn(
        "is_current",
        F.col("scd_valid_to").isNull()
    )
)


# Surrogate key
dim_customer = (
    customer_history
    .withColumn(
        "customer_key",
        F.monotonically_increasing_id()
    )
    .select(
        "customer_key",
        "customer_id",
        "full_name",
        "email",
        "country_code",
        "signup_date",
        "scd_valid_from",
        "scd_valid_to",
        "is_current"
    )
)


dim_customer.show(20, truncate=False)

+------------+-----------+-------------+------------------------+------------+-----------+-------------------+------------+----------+
|customer_key|customer_id|full_name    |email                   |country_code|signup_date|scd_valid_from     |scd_valid_to|is_current|
+------------+-----------+-------------+------------------------+------------+-----------+-------------------+------------+----------+
|0           |148        |Customer 148 |customer148@example.com |KSA         |2026-01-01 |2026-02-04 00:01:00|null        |true      |
|1           |463        |Customer 463 |customer463@example.com |UAE         |2026-01-01 |2026-02-07 16:24:00|null        |true      |
|2           |471        |Customer 471 |customer471@example.com |EG          |2026-01-01 |2026-05-21 17:50:00|null        |true      |
|3           |496        |Customer 496 |customer496@example.com |EG          |2026-01-01 |2026-05-16 03:42:00|null        |true      |
|4           |833        |Customer 833 |customer833@exa

In [166]:
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy("updated_at")
)


product_history = (
    products_df
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
)


product_history = (
    product_history
    .withColumn(
        "previous_product_name",
        F.lag("product_name").over(product_window)
    )
    .withColumn(
        "previous_category",
        F.lag("category").over(product_window)
    )
    .withColumn(
        "previous_unit_cost",
        F.lag("unit_cost").over(product_window)
    )
    .withColumn(
        "previous_list_price",
        F.lag("list_price").over(product_window)
    )
)


product_history = (
    product_history
    .withColumn(
        "is_first_version",
        F.col("previous_product_name").isNull()
        & F.col("previous_category").isNull()
        & F.col("previous_unit_cost").isNull()
        & F.col("previous_list_price").isNull()
    )
    .withColumn(
        "has_changed",
        (
            ~F.col("product_name").eqNullSafe(
                F.col("previous_product_name")
            )
            |
            ~F.col("category").eqNullSafe(
                F.col("previous_category")
            )
            |
            ~F.col("unit_cost").eqNullSafe(
                F.col("previous_unit_cost")
            )
            |
            ~F.col("list_price").eqNullSafe(
                F.col("previous_list_price")
            )
        )
    )
    .filter(
        F.col("is_first_version")
        | F.col("has_changed")
    )
)


product_scd_window = (
    Window
    .partitionBy("product_id")
    .orderBy("updated_at")
)


dim_product = (
    product_history
    .withColumn(
        "scd_valid_from",
        F.col("updated_at")
    )
    .withColumn(
        "scd_valid_to",
        F.lead("updated_at").over(product_scd_window)
    )
    .withColumn(
        "is_current",
        F.col("scd_valid_to").isNull()
    )
    .withColumn(
        "product_key",
        F.monotonically_increasing_id()
    )
    .select(
        "product_key",
        "product_id",
        "sku",
        "product_name",
        "category",
        "unit_cost",
        "list_price",
        "scd_valid_from",
        "scd_valid_to",
        "is_current"
    )
)


dim_product.show(20, truncate=False)

+-----------+----------+----------+------------+-----------+---------+----------+-------------------+------------+----------+
|product_key|product_id|sku       |product_name|category   |unit_cost|list_price|scd_valid_from     |scd_valid_to|is_current|
+-----------+----------+----------+------------+-----------+---------+----------+-------------------+------------+----------+
|0          |148       |SKU-000148|Product 148 |Beauty     |150.0    |182.34    |2026-06-04 04:09:00|null        |true      |
|1          |463       |SKU-000463|Product 463 |Sports     |75.67    |115.96    |2026-07-20 17:19:00|null        |true      |
|2          |471       |SKU-000471|Product 471 |Home       |346.88   |517.37    |2026-05-31 00:04:00|null        |true      |
|3          |496       |SKU-000496|Product 496 |Home       |121.81   |170.86    |2026-02-02 22:22:00|null        |true      |
|4          |833       |SKU-000833|Product 833 |Beauty     |202.23   |328.02    |2026-06-01 11:36:00|null        |true

In [167]:
SILVER_DB = "retailpulse_silver"
SILVER_BASE_PATH = "hdfs://namenode:8020/data/retailpulse/silver"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")

silver_tables = [
    "customers", "products", "stores", "orders",
    "order_items", "payments", "fulfillment_events", "inventory_snapshots"
]

def register_silver_csv(table):
    path = f"{SILVER_BASE_PATH}/{table}"

    # أسماء الأعمدة من الـ header الموجود في الملفات
    cols_list = spark.read.option("header", "true").csv(path).columns
    cols = ",\n  ".join(f"`{c}` STRING" for c in cols_list)

    spark.sql(f"DROP TABLE IF EXISTS {SILVER_DB}.{table}")
    spark.sql(f"""
        CREATE EXTERNAL TABLE {SILVER_DB}.{table} (
          {cols}
        )
        ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
        WITH SERDEPROPERTIES (
          'separatorChar' = ',',
          'quoteChar'     = '"',
          'escapeChar'    = '\\\\'
        )
        STORED AS TEXTFILE
        LOCATION '{path}'
        TBLPROPERTIES ('skip.header.line.count'='1')
    """)
    print(f"✓ registered {SILVER_DB}.{table}")

for t in silver_tables:
    register_silver_csv(t)

✓ registered retailpulse_silver.customers
✓ registered retailpulse_silver.products
✓ registered retailpulse_silver.stores
✓ registered retailpulse_silver.orders
✓ registered retailpulse_silver.order_items
✓ registered retailpulse_silver.payments
✓ registered retailpulse_silver.fulfillment_events
✓ registered retailpulse_silver.inventory_snapshots


In [168]:
# ============================================================
# CELL 14 - Save dim_product + Register Hive External Table
# ============================================================

table_name = "dim_product"
path = f"{GOLD_BASE_PATH}/{table_name}"

# 1. Save DataFrame as CSV on HDFS
(
    dim_product.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)

# 2. Remove old Hive table if it exists
spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")

# 3. Build Hive schema from Spark DataFrame schema
columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_product.schema.fields
    ]
)

# 4. Register CSV location as Hive External Table
spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")

print(f"✓ {table_name} saved to: {path}")
print(f"✓ Hive table registered: {GOLD_DATABASE}.{table_name}")

✓ dim_product saved to: hdfs://namenode:8020/data/retailpulse/gold/dim_product
✓ Hive table registered: retailpulse_gold.dim_product


In [169]:
# ============================================================
# CELL - dim_store
# ============================================================

dim_store = (
    stores_df
    .select(
        "store_id",
        "store_name",
        "city",
        "country_code",
        "opened_at"
    )
    .dropDuplicates(["store_id"])
    .withColumn(
        "store_key",
        F.monotonically_increasing_id()
    )
    .select(
        "store_key",
        "store_id",
        "store_name",
        "city",
        "country_code",
        "opened_at"
    )
)


# ------------------------------------------------------------
# Save dim_store as CSV on HDFS
# ------------------------------------------------------------

table_name = "dim_store"
path = f"{GOLD_BASE_PATH}/{table_name}"

(
    dim_store.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)


# ------------------------------------------------------------
# Drop old Hive table if exists
# ------------------------------------------------------------

spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")


# ------------------------------------------------------------
# Build Hive schema from Spark DataFrame schema
# ------------------------------------------------------------

columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_store.schema.fields
    ]
)


# ------------------------------------------------------------
# Create Hive External Table
# Spark 2.4.1 compatible
# ------------------------------------------------------------

spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print(f"✓ {table_name} saved to: {path}")
print(f"✓ Hive table created: {GOLD_DATABASE}.{table_name}")

dim_store.show(20, truncate=False)

✓ dim_store saved to: hdfs://namenode:8020/data/retailpulse/gold/dim_store
✓ Hive table created: retailpulse_gold.dim_store
+------------+--------+---------------------+-------+------------+-------------------+
|store_key   |store_id|store_name           |city   |country_code|opened_at          |
+------------+--------+---------------------+-------+------------+-------------------+
|17179869184 |31      |Retailpulse Store 031|City 11|EG          |2026-01-01 00:00:00|
|94489280512 |34      |Retailpulse Store 034|City 14|EG          |2026-01-01 00:00:00|
|120259084288|28      |Retailpulse Store 028|City 08|EG          |2026-01-01 00:00:00|
|163208757248|26      |Retailpulse Store 026|City 06|EG          |2026-01-01 00:00:00|
|163208757249|27      |Retailpulse Store 027|City 07|EG          |2026-01-01 00:00:00|
|180388626432|44      |Retailpulse Store 044|City 04|EG          |2026-01-01 00:00:00|
|206158430208|12      |Retailpulse Store 012|City 12|EG          |2026-01-01 00:00:00|
|25769

In [170]:
# ============================================================
# CELL 15 - Save dim_store + Register Hive External Table
# ============================================================

table_name = "dim_store"
path = f"{GOLD_BASE_PATH}/{table_name}"

# Save DataFrame as CSV
(
    dim_store.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)

# Drop old Hive table if exists
spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")

# Generate Hive schema from Spark DataFrame schema
columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_store.schema.fields
    ]
)

# Register as Hive External Table
spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")

print(f"✓ {table_name} saved to HDFS")
print(f"✓ Hive table registered: {GOLD_DATABASE}.{table_name}")

✓ dim_store saved to HDFS
✓ Hive table registered: retailpulse_gold.dim_store


In [171]:
# ============================================================
# CELL - dim_payment_method
# ============================================================

# ------------------------------------------------------------
# 1. Find latest payment for each order
# ------------------------------------------------------------

payment_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("paid_at").desc_nulls_last(),
        F.col("updated_at").desc_nulls_last()
    )
)


latest_payment = (
    payments_df
    .withColumn(
        "paid_at",
        F.to_timestamp("paid_at")
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "rn",
        F.row_number().over(payment_window)
    )
    .filter(
        F.col("rn") == 1
    )
)


# ------------------------------------------------------------
# 2. Build Payment Method Dimension
# ------------------------------------------------------------

dim_payment_method = (
    latest_payment
    .select(
        "payment_method",
        "payment_status"
    )
    .dropDuplicates()
    .withColumn(
        "payment_method_key",
        F.monotonically_increasing_id()
    )
    .select(
        "payment_method_key",
        "payment_method",
        "payment_status"
    )
)


# ------------------------------------------------------------
# 3. Save as CSV on HDFS
# ------------------------------------------------------------

table_name = "dim_payment_method"
path = f"{GOLD_BASE_PATH}/{table_name}"

(
    dim_payment_method.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)


# ------------------------------------------------------------
# 4. Drop old Hive table
# ------------------------------------------------------------

spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")


# ------------------------------------------------------------
# 5. Generate Hive schema
# ------------------------------------------------------------

columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_payment_method.schema.fields
    ]
)


# ------------------------------------------------------------
# 6. Create Hive External Table
# Spark 2.4.1 compatible
# ------------------------------------------------------------

spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")


# ------------------------------------------------------------
# 7. Verification
# ------------------------------------------------------------

print(f"✓ {table_name} saved to: {path}")
print(f"✓ Hive table registered: {GOLD_DATABASE}.{table_name}")

dim_payment_method.show(truncate=False)

✓ dim_payment_method saved to: hdfs://namenode:8020/data/retailpulse/gold/dim_payment_method
✓ Hive table registered: retailpulse_gold.dim_payment_method
+------------------+--------------+--------------+
|payment_method_key|payment_method|payment_status|
+------------------+--------------+--------------+
|85899345920       |card          |paid          |
|103079215104      |wallet        |captured      |
|335007449088      |wallet        |paid          |
|455266533376      |card          |captured      |
|549755813888      |cash          |captured      |
|652835028992      |wallet        |failed        |
|790273982464      |cash          |paid          |
|1056561954816     |card          |failed        |
|1408749273088     |cash          |failed        |
+------------------+--------------+--------------+



In [172]:
# ============================================================
# CELL 17 - dim_order_status
# ============================================================

dim_order_status = (
    orders_df
    .select(
        F.trim(F.lower(F.col("order_status"))).alias("order_status")
    )
    .dropDuplicates()
    .withColumn(
        "order_status_key",
        F.monotonically_increasing_id()
    )
    .select(
        "order_status_key",
        "order_status"
    )
)


# ------------------------------------------------------------
# Save as CSV on HDFS
# ------------------------------------------------------------

table_name = "dim_order_status"
path = f"{GOLD_BASE_PATH}/{table_name}"

(
    dim_order_status.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)


# ------------------------------------------------------------
# Drop old Hive table
# ------------------------------------------------------------

spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")


# ------------------------------------------------------------
# Generate Hive schema
# ------------------------------------------------------------

columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_order_status.schema.fields
    ]
)


# ------------------------------------------------------------
# Create Hive External Table
# ------------------------------------------------------------

spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")


print(f"✓ {table_name} saved to: {path}")
print(f"✓ Hive table registered")

dim_order_status.show(truncate=False)

✓ dim_order_status saved to: hdfs://namenode:8020/data/retailpulse/gold/dim_order_status
✓ Hive table registered
+----------------+------------+
|order_status_key|order_status|
+----------------+------------+
|214748364800    |completed   |
|309237645312    |shipped     |
|395136991232    |cancelled   |
+----------------+------------+



In [173]:
# ============================================================
# CELL 18 - dim_fulfillment_status
# ============================================================

fulfillment_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("event_timestamp").desc_nulls_last(),
        F.col("updated_at").desc_nulls_last()
    )
)


# ------------------------------------------------------------
# Get latest fulfillment event per order
# ------------------------------------------------------------

latest_fulfillment = (
    fulfillment_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "rn",
        F.row_number().over(fulfillment_window)
    )
    .filter(
        F.col("rn") == 1
    )
)


# ------------------------------------------------------------
# Build dimension
# ------------------------------------------------------------

dim_fulfillment_status = (
    latest_fulfillment
    .select(
        "event_type",
        "warehouse_code"
    )
    .dropDuplicates()
    .withColumn(
        "fulfillment_status_key",
        F.monotonically_increasing_id()
    )
    .select(
        "fulfillment_status_key",
        F.col("event_type").alias("latest_event_type"),
        "warehouse_code"
    )
)


# ------------------------------------------------------------
# Save as CSV
# ------------------------------------------------------------

table_name = "dim_fulfillment_status"
path = f"{GOLD_BASE_PATH}/{table_name}"

(
    dim_fulfillment_status.write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(path)
)


# ------------------------------------------------------------
# Drop old Hive table
# ------------------------------------------------------------

spark.sql(f"""
DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
""")


# ------------------------------------------------------------
# Generate Hive schema
# ------------------------------------------------------------

columns = ",\n".join(
    [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        for field in dim_fulfillment_status.schema.fields
    ]
)


# ------------------------------------------------------------
# Create Hive External Table
# ------------------------------------------------------------

spark.sql(f"""
CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
    {columns}
)
STORED AS TEXTFILE
LOCATION '{path}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
""")


print(f"✓ {table_name} saved to: {path}")
print(f"✓ Hive table registered")

dim_fulfillment_status.show(truncate=False)

✓ dim_fulfillment_status saved to: hdfs://namenode:8020/data/retailpulse/gold/dim_fulfillment_status
✓ Hive table registered
+----------------------+-----------------+--------------+
|fulfillment_status_key|latest_event_type|warehouse_code|
+----------------------+-----------------+--------------+
|300647710720          |delivered        |WH-3          |
|360777252864          |delivered        |WH-2          |
|1219770712064         |delivered        |WH-4          |
|1245540515840         |delivered        |WH-5          |
|1451698946048         |delivered        |WH-1          |
+----------------------+-----------------+--------------+



In [174]:
# ============================================================
# CELL 19 - fact_sales
# ============================================================

# ------------------------------------------------------------
# 1. Prepare order items + orders
# ------------------------------------------------------------

fact_base = (
    order_items_df.alias("oi")
    .join(
        orders_df.alias("o"),
        F.col("oi.order_id") == F.col("o.order_id"),
        "inner"
    )
    .select(
        F.col("oi.order_item_id"),
        F.col("oi.order_id"),
        F.col("oi.product_id"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        F.col("oi.line_discount"),

        F.col("o.customer_id"),
        F.col("o.store_id"),
        F.col("o.order_status"),
        F.col("o.order_timestamp"),
        F.col("o.discount_amount")
    )
)


# ------------------------------------------------------------
# 2. Calculate line amount
# ------------------------------------------------------------

fact_base = (
    fact_base
    .withColumn(
        "line_amount",
        (
            F.col("quantity") * F.col("unit_price")
            - F.col("line_discount")
        )
    )
)


# ------------------------------------------------------------
# 3. Calculate total line amount per order
# ------------------------------------------------------------

order_window = (
    Window
    .partitionBy("order_id")
)


fact_base = (
    fact_base
    .withColumn(
        "order_line_amount_total",
        F.sum("line_amount").over(order_window)
    )
)


# ------------------------------------------------------------
# 4. Allocate order discount to each order item
# ------------------------------------------------------------

fact_base = (
    fact_base
    .withColumn(
        "order_discount_allocated",
        F.when(
            F.col("order_line_amount_total") > 0,
            F.col("discount_amount")
            * F.col("line_amount")
            / F.col("order_line_amount_total")
        )
        .otherwise(F.lit(0))
    )
)

In [175]:
# ------------------------------------------------------------
# 7. Current Customer Dimension
# ------------------------------------------------------------

customer_lookup = (
    dim_customer
    .filter(
        F.col("is_current") == True
    )
    .select(
        "customer_id",
        "customer_key"
    )
)

In [176]:
# ------------------------------------------------------------
# 8. Current Product Dimension
# ------------------------------------------------------------

product_lookup = (
    dim_product
    .filter(
        F.col("is_current") == True
    )
    .select(
        "product_id",
        "product_key",
        "unit_cost"
    )
)

In [177]:
# ------------------------------------------------------------
# 9. Store Dimension
# ------------------------------------------------------------

store_lookup = (
    dim_store
    .select(
        "store_id",
        "store_key"
    )
)

In [178]:
# ------------------------------------------------------------
# 10. Payment Dimension
# ------------------------------------------------------------

latest_payment_lookup = (
    latest_payment
    .select(
        "order_id",
        "payment_method",
        "payment_status"
    )
)

In [179]:
# ------------------------------------------------------------
# 11. Latest Fulfillment per Order
# ------------------------------------------------------------

latest_fulfillment_lookup = (
    latest_fulfillment
    .select(
        "order_id",
        "event_type",
        "warehouse_code"
    )
)

In [180]:
# ------------------------------------------------------------
# 12. Join Customer Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        customer_lookup,
        "customer_id",
        "left"
    )
)


# ------------------------------------------------------------
# 13. Join Product Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        product_lookup,
        "product_id",
        "left"
    )
)


# ------------------------------------------------------------
# 14. Join Store Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        store_lookup,
        "store_id",
        "left"
    )
)


# ------------------------------------------------------------
# 15. Join Latest Payment
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        latest_payment_lookup,
        "order_id",
        "left"
    )
)

In [181]:
# ------------------------------------------------------------
# 16. Join Latest Fulfillment
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        latest_fulfillment_lookup,
        "order_id",
        "left"
    )
)

In [182]:
# ------------------------------------------------------------
# 17. Join Payment Method Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        dim_payment_method,
        ["payment_method", "payment_status"],
        "left"
    )
)


# ------------------------------------------------------------
# 18. Join Order Status Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        dim_order_status,
        "order_status",
        "left"
    )
)


# ------------------------------------------------------------
# 19. Join Fulfillment Status Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        dim_fulfillment_status,
        (
            (fact_base["event_type"] == dim_fulfillment_status["latest_event_type"])
            &
            (fact_base["warehouse_code"] == dim_fulfillment_status["warehouse_code"])
        ),
        "left"
    )
)

In [183]:
# ============================================================
# CELL 19 - Build Final Fact Sales
# Grain: One row per order_item
# ============================================================

# ------------------------------------------------------------
# 1. Base Fact
# order_items + orders
# ------------------------------------------------------------

fact_base = (
    order_items_df.alias("oi")
    .join(
        orders_df.alias("o"),
        F.col("oi.order_id") == F.col("o.order_id"),
        "inner"
    )
    .select(
        F.col("oi.order_item_id").alias("order_item_id"),
        F.col("oi.order_id").alias("order_id"),
        F.col("oi.product_id").alias("product_id"),
        F.col("oi.quantity").alias("quantity"),
        F.col("oi.unit_price").alias("unit_price"),
        F.col("oi.line_discount").alias("line_discount"),

        F.col("o.customer_id").alias("customer_id"),
        F.col("o.store_id").alias("store_id"),
        F.col("o.order_status").alias("order_status"),
        F.col("o.order_timestamp").alias("order_timestamp"),
        F.col("o.discount_amount").alias("discount_amount")
    )
)


# ------------------------------------------------------------
# 2. Calculate line amount
# line_amount = quantity * unit_price - line_discount
# ------------------------------------------------------------

fact_base = (
    fact_base
    .withColumn(
        "line_amount",
        (
            F.col("quantity") * F.col("unit_price")
            - F.col("line_discount")
        )
    )
)


# ------------------------------------------------------------
# 3. Calculate total line amount per order
# ------------------------------------------------------------

order_window = (
    Window
    .partitionBy("order_id")
)

fact_base = (
    fact_base
    .withColumn(
        "order_line_amount_total",
        F.sum("line_amount").over(order_window)
    )
)


# ------------------------------------------------------------
# 4. Allocate order-level discount to each line
# ------------------------------------------------------------

fact_base = (
    fact_base
    .withColumn(
        "order_discount_allocated",
        F.when(
            F.col("order_line_amount_total") > 0,
            F.col("discount_amount")
            * F.col("line_amount")
            / F.col("order_line_amount_total")
        )
        .otherwise(F.lit(0))
    )
)


# ------------------------------------------------------------
# 5. Calculate total payments per order
# ------------------------------------------------------------

payment_totals = (
    payments_df
    .groupBy("order_id")
    .agg(
        F.sum(
            F.col("amount")
        ).alias("payment_amount_total")
    )
)


# ------------------------------------------------------------
# 6. Join payment totals to fact
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        payment_totals,
        "order_id",
        "left"
    )
)


# ------------------------------------------------------------
# 7. Allocate payment amount to each order item
# ------------------------------------------------------------

fact_base = (
    fact_base
    .withColumn(
        "payment_amount_allocated",
        F.when(
            F.col("order_line_amount_total") > 0,
            F.coalesce(
                F.col("payment_amount_total"),
                F.lit(0)
            )
            * F.col("line_amount")
            / F.col("order_line_amount_total")
        )
        .otherwise(F.lit(0))
    )
)


# ------------------------------------------------------------
# 8. Current Customer Dimension
# ------------------------------------------------------------

customer_lookup = (
    dim_customer
    .filter(
        F.col("is_current") == True
    )
    .select(
        "customer_id",
        "customer_key"
    )
)


# ------------------------------------------------------------
# 9. Current Product Dimension
# ------------------------------------------------------------

product_lookup = (
    dim_product
    .filter(
        F.col("is_current") == True
    )
    .select(
        "product_id",
        "product_key",
        "unit_cost"
    )
)


# ------------------------------------------------------------
# 10. Store Dimension
# ------------------------------------------------------------

store_lookup = (
    dim_store
    .select(
        "store_id",
        "store_key"
    )
)


# ------------------------------------------------------------
# 11. Latest Payment per Order
# ------------------------------------------------------------

latest_payment_lookup = (
    latest_payment
    .select(
        "order_id",
        "payment_method",
        "payment_status"
    )
)


# ------------------------------------------------------------
# 12. Latest Fulfillment per Order
# ------------------------------------------------------------

latest_fulfillment_lookup = (
    latest_fulfillment
    .select(
        "order_id",
        "event_type",
        "warehouse_code"
    )
)


# ------------------------------------------------------------
# 13. Join Customer
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        customer_lookup,
        "customer_id",
        "left"
    )
)


# ------------------------------------------------------------
# 14. Join Product
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        product_lookup,
        "product_id",
        "left"
    )
)


# ------------------------------------------------------------
# 15. Join Store
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        store_lookup,
        "store_id",
        "left"
    )
)


# ------------------------------------------------------------
# 16. Join Latest Payment
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        latest_payment_lookup,
        "order_id",
        "left"
    )
)


# ------------------------------------------------------------
# 17. Join Latest Fulfillment
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        latest_fulfillment_lookup,
        "order_id",
        "left"
    )
)


# ------------------------------------------------------------
# 18. Join Payment Method Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        dim_payment_method,
        ["payment_method", "payment_status"],
        "left"
    )
)


# ------------------------------------------------------------
# 19. Join Order Status Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        dim_order_status,
        "order_status",
        "left"
    )
)


# ------------------------------------------------------------
# 20. Join Fulfillment Status Dimension
# ------------------------------------------------------------

fact_base = (
    fact_base
    .join(
        dim_fulfillment_status,
        (
            (fact_base["event_type"] ==
             dim_fulfillment_status["latest_event_type"])
            &
            (fact_base["warehouse_code"] ==
             dim_fulfillment_status["warehouse_code"])
        ),
        "left"
    )
)


# ------------------------------------------------------------
# 21. Build Final Fact
# ------------------------------------------------------------

fact_sales = (
    fact_base
    .withColumn(
        "sales_key",
        F.monotonically_increasing_id()
    )
    .withColumn(
        "date_key",
        F.date_format(
            F.to_date("order_timestamp"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "margin_amount",
        F.col("line_amount")
        - (
            F.col("unit_cost")
            * F.col("quantity")
        )
    )
    .select(
        "sales_key",

        "date_key",

        "customer_key",
        "product_key",
        "store_key",
        "payment_method_key",
        "order_status_key",
        "fulfillment_status_key",

        "order_id",
        "order_item_id",

        "quantity",
        "unit_price",
        "line_discount",

        "line_amount",
        "order_discount_allocated",
        "payment_amount_allocated",

        "unit_cost",
        "margin_amount"
    )
)


# ------------------------------------------------------------
# 22. Show Result
# ------------------------------------------------------------

print("✓ fact_sales created successfully")

fact_sales.printSchema()

fact_sales.show(
    20,
    truncate=False
)

✓ fact_sales created successfully
root
 |-- sales_key: long (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- store_key: long (nullable = true)
 |-- payment_method_key: long (nullable = true)
 |-- order_status_key: long (nullable = true)
 |-- fulfillment_status_key: long (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- line_discount: double (nullable = true)
 |-- line_amount: double (nullable = true)
 |-- order_discount_allocated: double (nullable = true)
 |-- payment_amount_allocated: double (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- margin_amount: double (nullable = true)

+------------+--------+-------------+-------------+------------+------------------+----------------+----------------------+--------+-------------+--------+

In [184]:
GOLD_BASE_PATH = "hdfs://namenode:8020/data/retailpulse/gold"
GOLD_DB = "retailpulse_gold"

def write_csv_table(df, table, partition_cols=None):
    path = f"{GOLD_BASE_PATH}/{table}"
    part = set(partition_cols or [])

    w = (df.write.mode("overwrite")
           .format("csv")
           .option("header", "true"))
    if partition_cols:
        w = w.partitionBy(*partition_cols)
    w.save(path)

    cols = ",\n  ".join(
        f"`{f.name}` {f.dataType.simpleString().upper()}"
        for f in df.schema.fields if f.name not in part
    )
    part_ddl = ""
    if partition_cols:
        pcols = ", ".join(
            f"`{f.name}` {f.dataType.simpleString().upper()}"
            for f in df.schema.fields if f.name in part
        )
        part_ddl = f"PARTITIONED BY ({pcols})"

    spark.sql(f"DROP TABLE IF EXISTS {GOLD_DB}.{table}")
    spark.sql(f"""
        CREATE EXTERNAL TABLE {GOLD_DB}.{table} (
          {cols}
        )
        {part_ddl}
        ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
        STORED AS TEXTFILE
        LOCATION '{path}'
        TBLPROPERTIES ('skip.header.line.count'='1',
                       'serialization.null.format'='')
    """)
    if partition_cols:
        spark.sql(f"MSCK REPAIR TABLE {GOLD_DB}.{table}")

In [185]:
money = ["order_discount_allocated", "payment_amount_allocated", "margin_amount"]
fact_out = fact_sales
for c in money:
    fact_out = fact_out.withColumn(c, F.col(c).cast("decimal(18,2)"))

fact_out = (
    fact_out
    .withColumn("year",  (F.col("date_key") / 10000).cast("int"))
    .withColumn("month", ((F.col("date_key") % 10000) / 100).cast("int"))
)

assert fact_out.count() == order_items_df.count(), "grain broken: duplicate joins"

write_csv_table(fact_out, "fact_sales", ["year", "month"])

spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.fact_sales").show()
spark.sql(f"SHOW PARTITIONS {GOLD_DB}.fact_sales").show()

+--------+
|count(1)|
+--------+
|  298805|
+--------+

+------------------+
|         partition|
+------------------+
| year=2026/month=1|
|year=2026/month=10|
| year=2026/month=2|
| year=2026/month=3|
| year=2026/month=4|
| year=2026/month=5|
| year=2026/month=6|
| year=2026/month=7|
| year=2026/month=8|
| year=2026/month=9|
+------------------+



In [186]:
write_csv_table(dim_customer, "dim_customer")

In [187]:
tables = ["customers","fulfillment_events","inventory_snapshots",
          "order_items","orders","payments","products","stores","streaming_events"]

for t in tables:
    (spark.table(f"retailpulse_bronze.{t}")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", "true")
        .csv(f"hdfs://namenode:8020/data/retailpulse/exports/bronze/{t}"))
    print("✓", t)

✓ customers
✓ fulfillment_events
✓ inventory_snapshots
✓ order_items
✓ orders
✓ payments
✓ products
✓ stores
✓ streaming_events


In [188]:
# ============================================================
# CHECK GOLD DIMENSIONS
# ============================================================

tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "dim_payment_method",
    "dim_order_status",
    "dim_fulfillment_status"
]

for table in tables:
    count = spark.sql(
        "SELECT COUNT(*) AS cnt FROM retailpulse_gold.{}".format(table)
    ).collect()[0]["cnt"]

    print("{:<25} {}".format(table, count))

dim_customer              39615
dim_product               10200
dim_store                 93
dim_payment_method        18
dim_order_status          6
dim_fulfillment_status    10


In [189]:
spark.sql("""
    DESCRIBE DATABASE EXTENDED retailpulse_gold
""").show(truncate=False)

+-------------------------+-------------------------------------------------------+
|database_description_item|database_description_value                             |
+-------------------------+-------------------------------------------------------+
|Database Name            |retailpulse_gold                                       |
|Description              |                                                       |
|Location                 |file:/mnt/notebooks/spark-warehouse/retailpulse_gold.db|
|Properties               |                                                       |
+-------------------------+-------------------------------------------------------+



In [190]:
spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS")

'hdfs://namenode:8020'

In [192]:
# ============================================================
# TEST - Save dim_product as CSV on HDFS
# ============================================================

HDFS_PRODUCT_PATH = \
    "hdfs://namenode:8020/user/hive/warehouse/retailpulse_gold/dim_product"

In [193]:
spark.sql("""
    DROP TABLE IF EXISTS retailpulse_gold.dim_product
""")

DataFrame[]

In [194]:
dim_product.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(HDFS_PRODUCT_PATH)

In [195]:
spark.sql("""
    CREATE TABLE retailpulse_gold.dim_product
    USING CSV
    OPTIONS (
        header "true",
        delimiter ","
    )
    LOCATION 'hdfs://namenode:8020/user/hive/warehouse/retailpulse_gold/dim_product'
""")

DataFrame[]

In [196]:
spark.sql("""
    SELECT COUNT(*)
    FROM retailpulse_gold.dim_product
""").show()

+--------+
|count(1)|
+--------+
|   10000|
+--------+



In [197]:
spark.sql("""
    DESCRIBE FORMATTED retailpulse_gold.dim_product
""").show(100, truncate=False)

+----------------------------+---------------------------------------------------------------------+-------+
|col_name                    |data_type                                                            |comment|
+----------------------------+---------------------------------------------------------------------+-------+
|product_key                 |string                                                               |null   |
|product_id                  |string                                                               |null   |
|sku                         |string                                                               |null   |
|product_name                |string                                                               |null   |
|category                    |string                                                               |null   |
|unit_cost                   |string                                                               |null   |
|list_price        

In [198]:
spark.sql("""
    DESCRIBE EXTENDED retailpulse_gold.dim_product
""").show(100, truncate=False)

+----------------------------+---------------------------------------------------------------------+-------+
|col_name                    |data_type                                                            |comment|
+----------------------------+---------------------------------------------------------------------+-------+
|product_key                 |string                                                               |null   |
|product_id                  |string                                                               |null   |
|sku                         |string                                                               |null   |
|product_name                |string                                                               |null   |
|category                    |string                                                               |null   |
|unit_cost                   |string                                                               |null   |
|list_price        

In [199]:
spark.sql("""
    SHOW CREATE TABLE retailpulse_gold.dim_product
""").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|createtab_stmt                                                                                                                                                                                                                                                                                                                                                                                                                           |
+-----------------------------------------------------------------------------------------------------------------------------------------------

In [200]:
spark.read \
    .option("header", "true") \
    .csv("hdfs://namenode:8020/data/retailpulse/gold/dim_order_status") \
    .show(truncate=False)

+----------------+------------+
|order_status_key|order_status|
+----------------+------------+
|214748364800    |completed   |
|395136991232    |cancelled   |
|309237645312    |shipped     |
+----------------+------------+



In [201]:
# Re-write Gold CSV files WITHOUT headers
# This makes them compatible with Presto reading TEXTFILE.

gold_dfs = {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_store": dim_store,
    "dim_payment_method": dim_payment_method,
    "dim_order_status": dim_order_status,
    "dim_fulfillment_status": dim_fulfillment_status,
    "fact_sales": fact_sales
}

for table_name, df in gold_dfs.items():

    path = f"{GOLD_BASE_PATH}/{table_name}"

    print(f"Rewriting: {table_name}")

    (
        df.write
        .mode("overwrite")
        .format("csv")
        .option("header", "false")
        .save(path)
    )

    print(f"✓ {table_name} saved without header")

Rewriting: dim_date
✓ dim_date saved without header
Rewriting: dim_customer
✓ dim_customer saved without header
Rewriting: dim_product
✓ dim_product saved without header
Rewriting: dim_store
✓ dim_store saved without header
Rewriting: dim_payment_method
✓ dim_payment_method saved without header
Rewriting: dim_order_status
✓ dim_order_status saved without header
Rewriting: dim_fulfillment_status
✓ dim_fulfillment_status saved without header
Rewriting: fact_sales
✓ fact_sales saved without header


In [202]:
# Re-create Gold Hive external tables
# Spark 2.4.1 compatible

for table_name, df in gold_dfs.items():

    path = f"{GOLD_BASE_PATH}/{table_name}"

    # Drop old Hive table
    spark.sql(f"""
        DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
    """)

    # Build Hive schema from Spark DataFrame schema
    columns = ",\n".join(
        [
            f"`{field.name}` {field.dataType.simpleString().upper()}"
            for field in df.schema.fields
        ]
    )

    # Create external Hive table
    spark.sql(f"""
        CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
            {columns}
        )
        STORED AS TEXTFILE
        LOCATION '{path}'
    """)

    print(f"✓ Hive table recreated: {table_name}")

✓ Hive table recreated: dim_date
✓ Hive table recreated: dim_customer
✓ Hive table recreated: dim_product
✓ Hive table recreated: dim_store
✓ Hive table recreated: dim_payment_method
✓ Hive table recreated: dim_order_status
✓ Hive table recreated: dim_fulfillment_status
✓ Hive table recreated: fact_sales


In [203]:
spark.sql("""
SELECT *
FROM retailpulse_gold.dim_order_status
""").show(truncate=False)


+----------------+------------+
|order_status_key|order_status|
+----------------+------------+
|null            |null        |
|null            |null        |
|null            |null        |
+----------------+------------+



In [204]:
# Re-create Gold Hive external tables
# CSV files have NO header and comma-separated fields.

for table_name, df in gold_dfs.items():

    path = f"{GOLD_BASE_PATH}/{table_name}"

    # Drop old Hive table
    spark.sql(f"""
        DROP TABLE IF EXISTS {GOLD_DATABASE}.{table_name}
    """)

    # Generate Hive schema from Spark DataFrame
    columns = ",\n".join(
        [
            f"`{field.name}` {field.dataType.simpleString().upper()}"
            for field in df.schema.fields
        ]
    )

    # Create Hive external table for comma-separated CSV
    spark.sql(f"""
        CREATE EXTERNAL TABLE {GOLD_DATABASE}.{table_name} (
            {columns}
        )
        ROW FORMAT DELIMITED
        FIELDS TERMINATED BY ','
        STORED AS TEXTFILE
        LOCATION '{path}'
    """)

    print(f"✓ Hive table recreated: {table_name}")

✓ Hive table recreated: dim_date
✓ Hive table recreated: dim_customer
✓ Hive table recreated: dim_product
✓ Hive table recreated: dim_store
✓ Hive table recreated: dim_payment_method
✓ Hive table recreated: dim_order_status
✓ Hive table recreated: dim_fulfillment_status
✓ Hive table recreated: fact_sales


In [205]:
spark.sql("""
SELECT *
FROM retailpulse_gold.dim_order_status
""").show(truncate=False)

+----------------+------------+
|order_status_key|order_status|
+----------------+------------+
|214748364800    |completed   |
|309237645312    |shipped     |
|395136991232    |cancelled   |
+----------------+------------+



In [ ]:
def cnt(x): return x.count()

# ---- 1. أعداد الصفوف ----
print("fact_sales      :", cnt(fact_sales), "| silver order_items:", cnt(order_items_df))
print("dim_store       :", cnt(dim_store), "| silver stores     :", cnt(stores_df))
print("dim_customer    :", cnt(dim_customer), "| silver customers  :", cnt(customers_df),
      "| current:", dim_customer.filter("is_current").count())
print("dim_product     :", cnt(dim_product), "| silver products   :", cnt(products_df),
      "| current:", dim_product.filter("is_current").count())
print("order_items بلا order:", order_items_df.join(orders_df, "order_id", "left_anti").count())

# ---- 2. المجاميع المالية ----
li = order_items_df.agg(F.sum(F.col("quantity")*F.col("unit_price") - F.col("line_discount"))).collect()[0][0]
fa = fact_sales.agg(F.sum("line_amount")).collect()[0][0]
print("line amount  silver:", li, " gold:", fa)

od_silver = orders_df.join(order_items_df.select("order_id").distinct(), "order_id") \
                     .agg(F.sum("discount_amount")).collect()[0][0]
od_gold = fact_sales.agg(F.sum("order_discount_allocated")).collect()[0][0]
print("order discount silver:", od_silver, " gold:", od_gold)

# ---- 3. مفاتيح يتيمة (NULL) ----
keys = ["customer_key","product_key","store_key","payment_method_key",
        "order_status_key","fulfillment_status_key"]
fact_sales.select([F.sum(F.col(k).isNull().cast("int")).alias(k) for k in keys]).show()

# ---- 4. تكرار ----
print("dup order_item_id:", cnt(fact_sales) - fact_sales.select("order_item_id").distinct())

fact_sales      : 298755 | silver order_items: 298755
dim_store       : 50 | silver stores     : 50


In [207]:
# ---- 4. تكرار ----
fact_cnt = fact_sales.count()
fact_distinct = fact_sales.select("order_item_id").distinct().count()
print("dup order_item_id:", fact_cnt - fact_distinct)

dup_current = (
    dim_customer.filter("is_current")
    .groupBy("customer_id").count()
    .filter("count > 1").count()
)
print("customers with >1 current row:", dup_current)

dup order_item_id: 0
customers with >1 current row: 0
